In [1]:
import pandas as pd


## Step 1: Load Dataset

In [7]:
df = pd.read_csv("HHS_Unaccompanied_Alien_Children_Program.csv")

In [8]:
# Display first few rows
print(df.head())

        Date  Children apprehended and placed in CBP custody*  \
0  21-Dec-25                                                6   
1  18-Dec-25                                               11   
2  17-Dec-25                                                7   
3  16-Dec-25                                                8   
4  15-Dec-25                                               11   

   Children in CBP custody  Children transferred out of CBP custody  \
0                       18                                       11   
1                       50                                        6   
2                       31                                       11   
3                       54                                       15   
4                       42                                        9   

  Children in HHS Care  Children discharged from HHS Care  Validation  
0                2,484                                 14        True  
1                2,472                

## Step 2: Convert Date to Datetime Format

In [9]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Check for invalid dates
print("Missing Dates after conversion:", df["Date"].isna().sum())

# Remove rows with invalid dates (optional)
df = df.dropna(subset=["Date"])

Missing Dates after conversion: 86


C:\Users\vp633\AppData\Local\Temp\ipykernel_13084\992188540.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")


## Step 3: Ensure Chronological Ordering

In [10]:
df = df.sort_values(by="Date").reset_index(drop=True)

## Step 4: Create Complete Daily Index

In [11]:
# Generate complete date range
date_range = pd.date_range(
    start=df["Date"].min(),
    end=df["Date"].max(),
    freq="D"
)

# Reindex dataset to include every day
df = (
    df.set_index("Date")
      .reindex(date_range)
      .rename_axis("Date")
      .reset_index()
)

## Step 5: Handle Missing Values

In [12]:
# Fill numeric columns with 0
numeric_cols = df.select_dtypes(include="number").columns
df[numeric_cols] = df[numeric_cols].fillna(0)

# Fill categorical columns with Forward Fill
categorical_cols = df.select_dtypes(include="object").columns
df[categorical_cols] = df[categorical_cols].ffill()

C:\Users\vp633\AppData\Local\Temp\ipykernel_13084\70356274.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include="object").columns



## Step 6: Verify Dataset

In [13]:
print(df.info())
print(df.head())
print(df.tail())

<class 'pandas.DataFrame'>
RangeIndex: 1085 entries, 0 to 1084
Data columns (total 7 columns):
 #   Column                                           Non-Null Count  Dtype         
---  ------                                           --------------  -----         
 0   Date                                             1085 non-null   datetime64[us]
 1   Children apprehended and placed in CBP custody*  1085 non-null   float64       
 2   Children in CBP custody                          1085 non-null   float64       
 3   Children transferred out of CBP custody          1085 non-null   float64       
 4   Children in HHS Care                             1085 non-null   str           
 5   Children discharged from HHS Care                1085 non-null   float64       
 6   Validation                                       1085 non-null   object        
dtypes: datetime64[us](1), float64(4), object(1), str(1)
memory usage: 64.9+ KB
None
        Date  Children apprehended and placed in CBP cu

## Step 7: Save Cleaned Dataset

In [14]:
df.to_csv("HHS_Cleaned_Daily_Data.csv", index=False)

print("\nData Ingestion & Structuring Completed Successfully!")


Data Ingestion & Structuring Completed Successfully!
